# WESAD Dataset Exploration

## Multimodal Stress Detection from Physiological Signals

**Author:** urme-b  
**Date:** 2024  
**Dataset:** WESAD (Wearable Stress and Affect Detection)

---

## Overview

This notebook explores the **WESAD dataset**, a publicly available multimodal dataset for wearable stress and affect detection. The dataset was collected by Schmidt et al. (2018) and contains physiological signals from 15 subjects during controlled stress induction experiments.

### Dataset Highlights

| Property | Value |
|----------|-------|
| Subjects | 15 (S2-S17, excluding S1, S12) |
| Devices | RespiBAN chest + Empatica E4 wrist |
| Chest signals | ECG, EDA, EMG, TEMP, ACC, RESP (700 Hz) |
| Wrist signals | BVP, EDA, TEMP, ACC (4-64 Hz) |
| Conditions | Baseline, Stress (TSST), Amusement, Meditation |
| Duration | ~2 hours per subject |

### Notebook Objectives

1. **Load and parse** WESAD pickle files
2. **Explore data structure** (chest vs wrist modalities)
3. **Visualize physiological signals** (ECG, EDA, Temperature)
4. **Analyze class distribution** across conditions
5. **Calculate statistics** per experimental condition
6. **Document data quality issues** for preprocessing pipeline

---

## 1. Environment Setup

First, we import necessary libraries and configure the notebook environment.

In [ ]:
# Standard library imports
import os
import sys
import pickle
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Signal processing (for basic analysis)
from scipy import signal
from scipy import stats

# Add parent directory to path for local imports
sys.path.insert(0, str(Path.cwd().parent))

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14
})

# Color palette for conditions
CONDITION_COLORS = {
    'baseline': '#2ecc71',      # Green
    'stress': '#e74c3c',        # Red
    'amusement': '#3498db',     # Blue
    'meditation': '#9b59b6',    # Purple
    'not_defined': '#95a5a6'    # Gray
}

print("Environment configured successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Dataset Loading

### 2.1 WESAD Data Structure

Each subject's data is stored in a pickle file with the following structure:

```python
data = {
    'subject': 'Sx',           # Subject ID
    'signal': {
        'chest': {             # RespiBAN (700 Hz)
            'ACC': array,      # Acceleration (3-axis)
            'ECG': array,      # Electrocardiogram
            'EMG': array,      # Electromyogram
            'EDA': array,      # Electrodermal Activity
            'Temp': array,     # Temperature
            'Resp': array      # Respiration
        },
        'wrist': {             # Empatica E4
            'ACC': array,      # Acceleration (32 Hz)
            'BVP': array,      # Blood Volume Pulse (64 Hz)
            'EDA': array,      # Electrodermal Activity (4 Hz)
            'TEMP': array      # Temperature (4 Hz)
        }
    },
    'label': array             # Ground truth labels (700 Hz)
}
```

### 2.2 Label Encoding

| Label | Condition | Description |
|-------|-----------|-------------|
| 0 | Not defined | Transient/transition periods |
| 1 | Baseline | Neutral reading task |
| 2 | Stress | TSST (Trier Social Stress Test) |
| 3 | Amusement | Watching funny video clips |
| 4 | Meditation | Guided relaxation |
| 5-7 | Other | Ignored (recovery periods) |

In [ ]:
# Define paths and constants
WESAD_PATH = Path('../data/raw/WESAD')

# Valid subject IDs (S1 and S12 excluded from original dataset)
VALID_SUBJECTS = [
    'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9',
    'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17'
]

# Sampling frequencies
FS_CHEST = 700      # Hz - RespiBAN chest device
FS_WRIST_ACC = 32   # Hz - Empatica E4 accelerometer
FS_WRIST_BVP = 64   # Hz - Empatica E4 BVP
FS_WRIST_EDA = 4    # Hz - Empatica E4 EDA
FS_WRIST_TEMP = 4   # Hz - Empatica E4 temperature

# Label mapping
LABEL_MAP = {
    0: 'not_defined',
    1: 'baseline',
    2: 'stress',
    3: 'amusement',
    4: 'meditation',
    5: 'other',
    6: 'other',
    7: 'other'
}

print(f"WESAD path: {WESAD_PATH.absolute()}")
print(f"Path exists: {WESAD_PATH.exists()}")

In [ ]:
def load_subject_data(subject_id: str, data_path: Path = WESAD_PATH) -> Dict:
    pkl_path = data_path / subject_id / f"{subject_id}.pkl"
    
    if not pkl_path.exists():
        raise FileNotFoundError(f"Data file not found: {pkl_path}")
    
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    
    return {
        'subject': data['subject'],
        'chest': data['signal']['chest'],
        'wrist': data['signal']['wrist'],
        'label': data['label']
    }


def get_available_subjects(data_path: Path = WESAD_PATH) -> List[str]:
    available = []
    for subj in VALID_SUBJECTS:
        pkl_path = data_path / subj / f"{subj}.pkl"
        if pkl_path.exists():
            available.append(subj)
    return available


# Check dataset availability
available_subjects = get_available_subjects()

if not available_subjects:
    print("="*60)
    print("WESAD DATASET NOT FOUND")
    print("="*60)
    print(f"\nExpected path: {WESAD_PATH.absolute()}")
    print("\nPlease download the dataset from:")
    print("https://archive.ics.uci.edu/ml/datasets/WESAD")
    print("\nSee data/raw/README.md for detailed instructions.")
    print("="*60)
    
    # Create synthetic demo data for notebook demonstration
    DEMO_MODE = True
    print("\n*** Running in DEMO MODE with synthetic data ***")
else:
    DEMO_MODE = False
    print(f"Found {len(available_subjects)} subjects: {available_subjects}")

In [ ]:
def generate_synthetic_data(duration_seconds: int = 7200) -> Dict:
    np.random.seed(42)
    
    n_samples_chest = duration_seconds * FS_CHEST
    n_samples_wrist_eda = duration_seconds * FS_WRIST_EDA
    n_samples_wrist_bvp = duration_seconds * FS_WRIST_BVP
    n_samples_wrist_acc = duration_seconds * FS_WRIST_ACC
    
    t_chest = np.arange(n_samples_chest) / FS_CHEST
    
    # Generate ECG-like signal (simplified QRS complexes)
    heart_rate = 70  # bpm
    ecg = np.zeros(n_samples_chest)
    beat_interval = int(FS_CHEST * 60 / heart_rate)
    for i in range(0, n_samples_chest, beat_interval):
        if i + 50 < n_samples_chest:
            # P wave
            ecg[i:i+20] = 0.1 * np.sin(np.linspace(0, np.pi, 20))
            # QRS complex
            ecg[i+25:i+30] = -0.2 * np.sin(np.linspace(0, np.pi, 5))
            ecg[i+30:i+35] = 1.0 * np.sin(np.linspace(0, np.pi, 5))
            ecg[i+35:i+40] = -0.3 * np.sin(np.linspace(0, np.pi, 5))
            # T wave
            ecg[i+45:i+65] = 0.2 * np.sin(np.linspace(0, np.pi, 20))
    ecg += 0.05 * np.random.randn(n_samples_chest)  # Add noise
    
    # Generate EDA signal (tonic + phasic components)
    eda_tonic = 5 + 2 * np.sin(2 * np.pi * 0.001 * t_chest)  # Slow drift
    eda_phasic = np.zeros(n_samples_chest)
    # Add SCRs (Skin Conductance Responses)
    for _ in range(500):
        onset = np.random.randint(0, n_samples_chest - 3000)
        amplitude = np.random.uniform(0.1, 1.0)
        rise = np.linspace(0, amplitude, 700)  # 1 second rise
        fall = amplitude * np.exp(-np.linspace(0, 3, 2000))  # Decay
        eda_phasic[onset:onset+700] += rise
        eda_phasic[onset+700:onset+2700] += fall
    eda = eda_tonic + eda_phasic + 0.01 * np.random.randn(n_samples_chest)
    eda = np.maximum(eda, 0)  # EDA is always positive
    
    # Generate temperature signal
    temp = 32 + 2 * np.sin(2 * np.pi * 0.0001 * t_chest)  # Slow variation
    temp += 0.1 * np.random.randn(n_samples_chest)
    
    # Generate EMG signal
    emg = 0.5 * np.random.randn(n_samples_chest)
    # Add muscle activity bursts
    for _ in range(200):
        onset = np.random.randint(0, n_samples_chest - 7000)
        duration = np.random.randint(3500, 7000)
        emg[onset:onset+duration] *= np.random.uniform(2, 5)
    
    # Generate respiration signal
    resp_rate = 15  # breaths per minute
    resp = np.sin(2 * np.pi * resp_rate/60 * t_chest)
    resp += 0.1 * np.random.randn(n_samples_chest)
    
    # Generate accelerometer (3-axis)
    acc_chest = np.column_stack([
        0.1 * np.random.randn(n_samples_chest),  # X
        0.1 * np.random.randn(n_samples_chest),  # Y
        9.8 + 0.1 * np.random.randn(n_samples_chest)  # Z (gravity)
    ])
    
    # Generate labels (simulated protocol)
    labels = np.zeros(n_samples_chest, dtype=int)
    # Protocol: baseline -> stress -> baseline -> amusement -> meditation
    segments = [
        (0, 20*60, 1),           # 20 min baseline
        (20*60, 30*60, 0),       # 10 min transition
        (30*60, 50*60, 2),       # 20 min stress (TSST)
        (50*60, 55*60, 0),       # 5 min transition
        (55*60, 70*60, 1),       # 15 min baseline
        (70*60, 75*60, 0),       # 5 min transition
        (75*60, 90*60, 3),       # 15 min amusement
        (90*60, 95*60, 0),       # 5 min transition
        (95*60, 115*60, 4),      # 20 min meditation
        (115*60, 120*60, 0)      # 5 min transition
    ]
    for start, end, label in segments:
        start_idx = int(start * FS_CHEST)
        end_idx = min(int(end * FS_CHEST), n_samples_chest)
        labels[start_idx:end_idx] = label
    
    # Modify signals based on condition (stress increases HR, EDA, etc.)
    stress_mask = labels == 2
    eda[stress_mask] *= 1.5
    
    # Wrist signals (downsampled/different rates)
    wrist_eda = signal.resample(eda, n_samples_wrist_eda)
    wrist_temp = signal.resample(temp, n_samples_wrist_eda)
    wrist_bvp = np.sin(2 * np.pi * 1.2 * np.arange(n_samples_wrist_bvp) / FS_WRIST_BVP)
    wrist_bvp += 0.1 * np.random.randn(n_samples_wrist_bvp)
    wrist_acc = 0.1 * np.random.randn(n_samples_wrist_acc, 3)
    wrist_acc[:, 2] += 9.8
    
    return {
        'subject': 'S_DEMO',
        'chest': {
            'ECG': ecg.reshape(-1, 1),
            'EDA': eda.reshape(-1, 1),
            'EMG': emg.reshape(-1, 1),
            'Temp': temp.reshape(-1, 1),
            'Resp': resp.reshape(-1, 1),
            'ACC': acc_chest
        },
        'wrist': {
            'EDA': wrist_eda.reshape(-1, 1),
            'TEMP': wrist_temp.reshape(-1, 1),
            'BVP': wrist_bvp.reshape(-1, 1),
            'ACC': wrist_acc
        },
        'label': labels
    }


# Load or generate data
if DEMO_MODE:
    print("Generating synthetic data for demonstration...")
    subject_data = generate_synthetic_data(duration_seconds=7200)  # 2 hours
    SUBJECT_ID = 'S_DEMO'
else:
    SUBJECT_ID = available_subjects[0]  # Load first available subject
    print(f"Loading subject {SUBJECT_ID}...")
    subject_data = load_subject_data(SUBJECT_ID)

print(f"\nData loaded for subject: {subject_data['subject']}")

## 3. Data Structure Exploration

Let's examine the structure and dimensions of the loaded data to understand what we're working with.

In [ ]:
def explore_data_structure(data: Dict) -> pd.DataFrame:
    rows = []
    
    # Chest signals
    for signal_name, signal_data in data['chest'].items():
        signal_data = np.array(signal_data)
        rows.append({
            'Device': 'Chest (RespiBAN)',
            'Signal': signal_name,
            'Shape': str(signal_data.shape),
            'Samples': signal_data.shape[0],
            'Channels': signal_data.shape[1] if signal_data.ndim > 1 else 1,
            'Sampling Rate': f"{FS_CHEST} Hz",
            'Duration (min)': f"{signal_data.shape[0] / FS_CHEST / 60:.1f}",
            'Min': f"{np.min(signal_data):.3f}",
            'Max': f"{np.max(signal_data):.3f}",
            'Mean': f"{np.mean(signal_data):.3f}",
            'Std': f"{np.std(signal_data):.3f}"
        })
    
    # Wrist signals
    wrist_fs_map = {'ACC': FS_WRIST_ACC, 'BVP': FS_WRIST_BVP, 'EDA': FS_WRIST_EDA, 'TEMP': FS_WRIST_EDA}
    for signal_name, signal_data in data['wrist'].items():
        signal_data = np.array(signal_data)
        fs = wrist_fs_map.get(signal_name, FS_WRIST_EDA)
        rows.append({
            'Device': 'Wrist (Empatica E4)',
            'Signal': signal_name,
            'Shape': str(signal_data.shape),
            'Samples': signal_data.shape[0],
            'Channels': signal_data.shape[1] if signal_data.ndim > 1 else 1,
            'Sampling Rate': f"{fs} Hz",
            'Duration (min)': f"{signal_data.shape[0] / fs / 60:.1f}",
            'Min': f"{np.min(signal_data):.3f}",
            'Max': f"{np.max(signal_data):.3f}",
            'Mean': f"{np.mean(signal_data):.3f}",
            'Std': f"{np.std(signal_data):.3f}"
        })
    
    # Labels
    labels = np.array(data['label'])
    rows.append({
        'Device': 'Ground Truth',
        'Signal': 'Labels',
        'Shape': str(labels.shape),
        'Samples': labels.shape[0],
        'Channels': 1,
        'Sampling Rate': f"{FS_CHEST} Hz",
        'Duration (min)': f"{labels.shape[0] / FS_CHEST / 60:.1f}",
        'Min': f"{np.min(labels):.0f}",
        'Max': f"{np.max(labels):.0f}",
        'Mean': f"{np.mean(labels):.2f}",
        'Std': f"{np.std(labels):.2f}"
    })
    
    return pd.DataFrame(rows)


# Display data structure
structure_df = explore_data_structure(subject_data)
print(f"\n{'='*80}")
print(f"DATA STRUCTURE SUMMARY - Subject {subject_data['subject']}")
print(f"{'='*80}\n")
display(structure_df)

### Key Observations

1. **Chest device (RespiBAN)**: All signals sampled at 700 Hz, providing high temporal resolution
2. **Wrist device (Empatica E4)**: Variable sampling rates (4-64 Hz) - requires resampling for fusion
3. **Accelerometer**: Both devices have 3-axis accelerometer data
4. **Labels**: Synchronized with chest signals at 700 Hz

---

## 4. Signal Visualization

### 4.1 Complete Recording Overview

First, let's visualize the entire recording with labels to understand the experimental protocol.

In [ ]:
def plot_full_recording(data: Dict, signals_to_plot: List[str] = ['ECG', 'EDA', 'Temp']):
    n_signals = len(signals_to_plot) + 1  # +1 for labels
    fig, axes = plt.subplots(n_signals, 1, figsize=(16, 3*n_signals), sharex=True)
    
    labels = np.array(data['label']).flatten()
    n_samples = len(labels)
    time_minutes = np.arange(n_samples) / FS_CHEST / 60
    
    # Downsample for faster plotting (every 100th sample)
    downsample = 100
    time_ds = time_minutes[::downsample]
    labels_ds = labels[::downsample]
    
    # Plot each signal
    for idx, signal_name in enumerate(signals_to_plot):
        ax = axes[idx]
        signal = np.array(data['chest'][signal_name]).flatten()
        signal_ds = signal[::downsample]
        
        # Color by condition
        for label_val in [1, 2, 3, 4]:
            mask = labels_ds == label_val
            if np.any(mask):
                condition = LABEL_MAP[label_val]
                ax.scatter(time_ds[mask], signal_ds[mask], 
                          c=CONDITION_COLORS[condition], s=0.1, alpha=0.7,
                          label=condition.capitalize())
        
        ax.set_ylabel(signal_name)
        ax.set_title(f'{signal_name} Signal (colored by condition)')
        
        if idx == 0:
            ax.legend(loc='upper right', markerscale=20)
    
    # Plot labels
    ax = axes[-1]
    for label_val in range(5):
        mask = labels_ds == label_val
        if np.any(mask):
            condition = LABEL_MAP[label_val]
            ax.fill_between(time_ds, 0, 1, where=mask,
                           color=CONDITION_COLORS[condition], alpha=0.7,
                           label=condition.capitalize())
    
    ax.set_ylabel('Condition')
    ax.set_xlabel('Time (minutes)')
    ax.set_title('Experimental Protocol Timeline')
    ax.set_yticks([])
    ax.legend(loc='upper right', ncol=5)
    
    plt.suptitle(f'Full Recording Overview - Subject {data["subject"]}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


# Plot full recording
plot_full_recording(subject_data, signals_to_plot=['ECG', 'EDA', 'Temp'])

### 4.2 Detailed Signal Segments

Now let's zoom into specific segments to examine signal quality and characteristics.

In [ ]:
def plot_signal_segment(data: Dict, signal_name: str, start_sec: float, 
                        duration_sec: float = 10, device: str = 'chest'):
    if device == 'chest':
        fs = FS_CHEST
        signal = np.array(data['chest'][signal_name]).flatten()
    else:
        fs_map = {'ACC': FS_WRIST_ACC, 'BVP': FS_WRIST_BVP, 'EDA': FS_WRIST_EDA, 'TEMP': FS_WRIST_EDA}
        fs = fs_map.get(signal_name, FS_WRIST_EDA)
        signal = np.array(data['wrist'][signal_name]).flatten()
    
    start_idx = int(start_sec * fs)
    end_idx = int((start_sec + duration_sec) * fs)
    
    segment = signal[start_idx:end_idx]
    time = np.arange(len(segment)) / fs
    
    # Get label for this segment (from chest labels)
    label_start = int(start_sec * FS_CHEST)
    label_end = int((start_sec + duration_sec) * FS_CHEST)
    labels = np.array(data['label']).flatten()
    segment_labels = labels[label_start:label_end]
    dominant_label = int(stats.mode(segment_labels, keepdims=False)[0])
    condition = LABEL_MAP[dominant_label]
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(time, segment, color=CONDITION_COLORS.get(condition, '#333'), linewidth=0.8)
    ax.fill_between(time, segment.min(), segment, alpha=0.3, 
                    color=CONDITION_COLORS.get(condition, '#333'))
    
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel(f'{signal_name} Value')
    ax.set_title(f'{signal_name} Signal ({device.capitalize()}) - '
                 f'{duration_sec}s segment starting at {start_sec/60:.1f} min\n'
                 f'Condition: {condition.upper()}', fontsize=12)
    
    # Add statistics box
    stats_text = f'Mean: {np.mean(segment):.3f}\nStd: {np.std(segment):.3f}\n'
    stats_text += f'Min: {np.min(segment):.3f}\nMax: {np.max(segment):.3f}'
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    return segment


# Plot ECG segment during baseline (first 5 minutes)
print("ECG Signal - 10 second segment during BASELINE condition:")
ecg_segment = plot_signal_segment(subject_data, 'ECG', start_sec=300, duration_sec=10)

In [ ]:
# Plot ECG segment during stress (around 35 minutes)
print("ECG Signal - 10 second segment during STRESS condition:")
ecg_stress = plot_signal_segment(subject_data, 'ECG', start_sec=2100, duration_sec=10)

In [ ]:
# Plot EDA signal showing SCRs
print("EDA Signal - 60 second segment showing Skin Conductance Responses (SCRs):")
eda_segment = plot_signal_segment(subject_data, 'EDA', start_sec=2000, duration_sec=60)

In [ ]:
# Plot temperature signal
print("Temperature Signal - 5 minute segment:")
temp_segment = plot_signal_segment(subject_data, 'Temp', start_sec=1800, duration_sec=300)

### 4.3 Multi-Signal Comparison Across Conditions

Let's compare how different signals look across experimental conditions.

In [ ]:
def plot_condition_comparison(data: Dict, signal_name: str, segment_duration: float = 30):
    labels = np.array(data['label']).flatten()
    signal = np.array(data['chest'][signal_name]).flatten()
    
    conditions = ['baseline', 'stress', 'amusement', 'meditation']
    label_values = [1, 2, 3, 4]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.flatten()
    
    for idx, (condition, label_val) in enumerate(zip(conditions, label_values)):
        ax = axes[idx]
        
        # Find indices for this condition
        condition_indices = np.where(labels == label_val)[0]
        
        if len(condition_indices) == 0:
            ax.text(0.5, 0.5, f'No data for {condition}', 
                    transform=ax.transAxes, ha='center')
            continue
        
        # Get a segment from the middle of this condition
        mid_idx = condition_indices[len(condition_indices)//2]
        segment_samples = int(segment_duration * FS_CHEST)
        start_idx = max(0, mid_idx - segment_samples//2)
        end_idx = min(len(signal), start_idx + segment_samples)
        
        segment = signal[start_idx:end_idx]
        time = np.arange(len(segment)) / FS_CHEST
        
        ax.plot(time, segment, color=CONDITION_COLORS[condition], linewidth=0.8)
        ax.fill_between(time, segment.min(), segment, 
                        color=CONDITION_COLORS[condition], alpha=0.3)
        
        ax.set_title(f'{condition.upper()}\nMean: {np.mean(segment):.3f}, Std: {np.std(segment):.3f}')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel(signal_name)
    
    plt.suptitle(f'{signal_name} Signal Comparison Across Conditions\n'
                 f'Subject {data["subject"]} - {segment_duration}s segments', fontsize=13)
    plt.tight_layout()
    plt.show()


# Compare ECG across conditions
plot_condition_comparison(subject_data, 'ECG', segment_duration=10)

In [ ]:
# Compare EDA across conditions
plot_condition_comparison(subject_data, 'EDA', segment_duration=60)

---

## 5. Class Distribution Analysis

Understanding the distribution of labels is crucial for handling class imbalance in machine learning.

In [ ]:
def analyze_label_distribution(data: Dict) -> pd.DataFrame:
    labels = np.array(data['label']).flatten()
    total_samples = len(labels)
    
    rows = []
    for label_val in sorted(np.unique(labels)):
        count = np.sum(labels == label_val)
        duration_sec = count / FS_CHEST
        duration_min = duration_sec / 60
        percentage = count / total_samples * 100
        
        rows.append({
            'Label': int(label_val),
            'Condition': LABEL_MAP.get(label_val, 'unknown').capitalize(),
            'Samples': f"{count:,}",
            'Duration (sec)': f"{duration_sec:.1f}",
            'Duration (min)': f"{duration_min:.1f}",
            'Percentage': f"{percentage:.1f}%"
        })
    
    # Add total row
    rows.append({
        'Label': '-',
        'Condition': 'TOTAL',
        'Samples': f"{total_samples:,}",
        'Duration (sec)': f"{total_samples/FS_CHEST:.1f}",
        'Duration (min)': f"{total_samples/FS_CHEST/60:.1f}",
        'Percentage': '100%'
    })
    
    return pd.DataFrame(rows)


# Analyze label distribution
label_dist = analyze_label_distribution(subject_data)
print(f"\n{'='*60}")
print(f"LABEL DISTRIBUTION - Subject {subject_data['subject']}")
print(f"{'='*60}\n")
display(label_dist)

In [ ]:
def plot_label_distribution(data: Dict):
    labels = np.array(data['label']).flatten()
    
    # Only include conditions we care about (1-4)
    conditions = ['Baseline', 'Stress', 'Amusement', 'Meditation']
    label_values = [1, 2, 3, 4]
    counts = [np.sum(labels == lv) for lv in label_values]
    colors = [CONDITION_COLORS[LABEL_MAP[lv]] for lv in label_values]
    
    # Calculate durations in minutes
    durations = [c / FS_CHEST / 60 for c in counts]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Pie chart
    ax1 = axes[0]
    wedges, texts, autotexts = ax1.pie(counts, labels=conditions, colors=colors,
                                        autopct='%1.1f%%', startangle=90,
                                        explode=[0.02]*4)
    ax1.set_title('Class Distribution (Experimental Conditions Only)')
    
    # Bar chart
    ax2 = axes[1]
    bars = ax2.bar(conditions, durations, color=colors, edgecolor='black', linewidth=1.2)
    ax2.set_ylabel('Duration (minutes)')
    ax2.set_title('Time Spent in Each Condition')
    
    # Add value labels on bars
    for bar, duration in zip(bars, durations):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{duration:.1f} min', ha='center', va='bottom', fontsize=10)
    
    plt.suptitle(f'Label Distribution Analysis - Subject {data["subject"]}', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Print class imbalance ratio
    max_count = max(counts)
    min_count = min(counts)
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
    
    print(f"\nClass Imbalance Analysis:")
    print(f"  Largest class: {conditions[counts.index(max_count)]} ({max_count:,} samples)")
    print(f"  Smallest class: {conditions[counts.index(min_count)]} ({min_count:,} samples)")
    print(f"  Imbalance ratio: {imbalance_ratio:.2f}:1")


# Plot distribution
plot_label_distribution(subject_data)

### Class Distribution Insights

**Key Findings:**

1. **Not Defined (Label 0)**: Significant portion of data is transition/unlabeled periods - must be excluded
2. **Class Imbalance**: Some conditions have more data than others
3. **Binary Classification**: For stress detection, we typically use Baseline (1) vs Stress (2)
4. **Multiclass**: Can also include Amusement (3) as a third affective state

**Implications for ML:**
- Use stratified splitting to maintain class proportions
- Consider class weights or oversampling (SMOTE)
- Report F1-score and confusion matrix alongside accuracy

---

## 6. Statistics Per Condition

Let's calculate descriptive statistics for each signal across different experimental conditions.

In [ ]:
def calculate_condition_statistics(data: Dict, signals: List[str] = None) -> pd.DataFrame:
    if signals is None:
        signals = ['ECG', 'EDA', 'Temp', 'Resp']
    
    labels = np.array(data['label']).flatten()
    conditions = ['baseline', 'stress', 'amusement', 'meditation']
    label_values = [1, 2, 3, 4]
    
    rows = []
    
    for signal_name in signals:
        signal = np.array(data['chest'][signal_name]).flatten()
        
        for condition, label_val in zip(conditions, label_values):
            mask = labels == label_val
            if not np.any(mask):
                continue
                
            segment = signal[mask]
            
            rows.append({
                'Signal': signal_name,
                'Condition': condition.capitalize(),
                'Mean': np.mean(segment),
                'Std': np.std(segment),
                'Min': np.min(segment),
                'Max': np.max(segment),
                'Median': np.median(segment),
                'IQR': np.percentile(segment, 75) - np.percentile(segment, 25),
                'Skewness': stats.skew(segment),
                'Kurtosis': stats.kurtosis(segment)
            })
    
    df = pd.DataFrame(rows)
    
    # Round numeric columns
    numeric_cols = ['Mean', 'Std', 'Min', 'Max', 'Median', 'IQR', 'Skewness', 'Kurtosis']
    df[numeric_cols] = df[numeric_cols].round(4)
    
    return df


# Calculate statistics
condition_stats = calculate_condition_statistics(subject_data)
print(f"\n{'='*80}")
print(f"SIGNAL STATISTICS PER CONDITION - Subject {subject_data['subject']}")
print(f"{'='*80}\n")

# Display pivoted for better readability
for signal in ['ECG', 'EDA', 'Temp', 'Resp']:
    print(f"\n{signal} Signal:")
    signal_stats = condition_stats[condition_stats['Signal'] == signal].drop(columns=['Signal'])
    display(signal_stats)

In [ ]:
def plot_statistics_comparison(data: Dict, signal_name: str):
    labels = np.array(data['label']).flatten()
    signal = np.array(data['chest'][signal_name]).flatten()
    
    conditions = ['Baseline', 'Stress', 'Amusement', 'Meditation']
    label_values = [1, 2, 3, 4]
    
    # Sample data for box plots (full data is too large)
    np.random.seed(42)
    sample_size = 10000
    
    plot_data = []
    plot_labels = []
    
    for condition, label_val in zip(conditions, label_values):
        mask = labels == label_val
        if np.any(mask):
            segment = signal[mask]
            # Random sample for visualization
            if len(segment) > sample_size:
                segment = np.random.choice(segment, sample_size, replace=False)
            plot_data.append(segment)
            plot_labels.append(condition)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot
    ax1 = axes[0]
    bp = ax1.boxplot(plot_data, labels=plot_labels, patch_artist=True)
    
    colors = [CONDITION_COLORS[c.lower()] for c in plot_labels]
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax1.set_ylabel(f'{signal_name} Value')
    ax1.set_title(f'{signal_name} Distribution by Condition\n(Box Plot)')
    ax1.grid(True, alpha=0.3)
    
    # Violin plot
    ax2 = axes[1]
    parts = ax2.violinplot(plot_data, positions=range(len(plot_labels)), showmeans=True, showmedians=True)
    
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
    
    ax2.set_xticks(range(len(plot_labels)))
    ax2.set_xticklabels(plot_labels)
    ax2.set_ylabel(f'{signal_name} Value')
    ax2.set_title(f'{signal_name} Distribution by Condition\n(Violin Plot)')
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle(f'Statistical Comparison - Subject {data["subject"]}', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


# Plot comparisons
plot_statistics_comparison(subject_data, 'EDA')

In [ ]:
plot_statistics_comparison(subject_data, 'ECG')

In [ ]:
plot_statistics_comparison(subject_data, 'Temp')

### Statistical Insights

**EDA (Electrodermal Activity):**
- Higher mean and variance during **stress** - indicates increased sympathetic nervous system activity
- More SCRs (Skin Conductance Responses) during stress
- Amusement also shows elevated EDA (positive arousal)

**ECG:**
- Signal morphology relatively consistent across conditions
- Derived HRV features will show more differentiation
- Heart rate increases during stress (visible in R-R intervals)

**Temperature:**
- Peripheral temperature decreases during stress (vasoconstriction)
- Meditation shows more stable temperature patterns

---

## 7. Data Quality Assessment

Identifying data quality issues is critical for preprocessing pipeline design.

In [ ]:
def assess_data_quality(data: Dict) -> Dict:
    quality_report = {}
    
    for device in ['chest', 'wrist']:
        for signal_name, signal_data in data[device].items():
            signal = np.array(signal_data).flatten()
            key = f"{device}_{signal_name}"
            
            # Basic checks
            n_samples = len(signal)
            n_nan = np.sum(np.isnan(signal))
            n_inf = np.sum(np.isinf(signal))
            
            # Outlier detection
            mean, std = np.nanmean(signal), np.nanstd(signal)
            outlier_mask = np.abs(signal - mean) > 5 * std
            n_outliers = np.sum(outlier_mask)
            
            # Flat segment detection (variance near zero)
            window_size = 100
            n_flat = 0
            for i in range(0, len(signal) - window_size, window_size):
                if np.std(signal[i:i+window_size]) < 1e-6:
                    n_flat += window_size
            
            # Signal-to-noise ratio estimate (simplified)
            # Using median filter to estimate signal, remainder is noise
            try:
                from scipy.ndimage import median_filter
                smoothed = median_filter(signal[:10000], size=51)  # Sample for speed
                noise = signal[:10000] - smoothed
                snr = 10 * np.log10(np.var(smoothed) / (np.var(noise) + 1e-10))
            except:
                snr = np.nan
            
            quality_report[key] = {
                'total_samples': n_samples,
                'nan_count': n_nan,
                'nan_percent': n_nan / n_samples * 100,
                'inf_count': n_inf,
                'outlier_count': n_outliers,
                'outlier_percent': n_outliers / n_samples * 100,
                'flat_samples': n_flat,
                'flat_percent': n_flat / n_samples * 100,
                'snr_db': snr,
                'min_value': np.nanmin(signal),
                'max_value': np.nanmax(signal),
                'dynamic_range': np.nanmax(signal) - np.nanmin(signal)
            }
    
    return quality_report


# Run quality assessment
quality_report = assess_data_quality(subject_data)

# Display as DataFrame
quality_df = pd.DataFrame(quality_report).T
quality_df = quality_df.round(3)

print(f"\n{'='*80}")
print(f"DATA QUALITY REPORT - Subject {subject_data['subject']}")
print(f"{'='*80}\n")
display(quality_df)

In [ ]:
def visualize_quality_issues(data: Dict):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. ECG with potential artifacts highlighted
    ax = axes[0, 0]
    ecg = np.array(data['chest']['ECG']).flatten()
    sample_start = 100000
    sample_end = 110000
    ecg_segment = ecg[sample_start:sample_end]
    time = np.arange(len(ecg_segment)) / FS_CHEST
    
    mean, std = np.mean(ecg_segment), np.std(ecg_segment)
    outlier_mask = np.abs(ecg_segment - mean) > 3 * std
    
    ax.plot(time, ecg_segment, 'b-', linewidth=0.5, label='ECG')
    ax.scatter(time[outlier_mask], ecg_segment[outlier_mask], c='red', s=10, label='Potential artifacts')
    ax.axhline(y=mean + 3*std, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=mean - 3*std, color='r', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('ECG')
    ax.set_title('ECG Signal with Potential Artifacts (>3σ)')
    ax.legend()
    
    # 2. EDA baseline drift
    ax = axes[0, 1]
    eda = np.array(data['chest']['EDA']).flatten()
    # Downsample for plotting
    eda_ds = eda[::100]
    time_ds = np.arange(len(eda_ds)) / (FS_CHEST/100) / 60
    
    # Compute baseline (very low-pass filter)
    from scipy.ndimage import uniform_filter1d
    baseline = uniform_filter1d(eda_ds, size=1000)
    
    ax.plot(time_ds, eda_ds, 'g-', alpha=0.5, linewidth=0.5, label='EDA')
    ax.plot(time_ds, baseline, 'r-', linewidth=2, label='Baseline drift')
    ax.set_xlabel('Time (minutes)')
    ax.set_ylabel('EDA (μS)')
    ax.set_title('EDA Signal with Baseline Drift')
    ax.legend()
    
    # 3. Signal distribution (histogram)
    ax = axes[1, 0]
    signals_to_hist = ['ECG', 'EDA', 'Temp']
    colors = ['blue', 'green', 'orange']
    
    for sig_name, color in zip(signals_to_hist, colors):
        sig = np.array(data['chest'][sig_name]).flatten()
        # Normalize for comparison
        sig_norm = (sig - np.mean(sig)) / np.std(sig)
        ax.hist(sig_norm[::100], bins=100, alpha=0.5, label=sig_name, color=color, density=True)
    
    ax.set_xlabel('Normalized Value (z-score)')
    ax.set_ylabel('Density')
    ax.set_title('Signal Value Distributions')
    ax.legend()
    ax.set_xlim(-5, 5)
    
    # 4. Missing data / label coverage
    ax = axes[1, 1]
    labels = np.array(data['label']).flatten()
    
    # Plot label timeline
    time_min = np.arange(len(labels)) / FS_CHEST / 60
    labels_ds = labels[::1000]
    time_ds = time_min[::1000]
    
    for label_val in range(5):
        mask = labels_ds == label_val
        condition = LABEL_MAP[label_val]
        ax.scatter(time_ds[mask], labels_ds[mask], c=CONDITION_COLORS[condition], 
                   s=1, label=condition.capitalize(), alpha=0.7)
    
    ax.set_xlabel('Time (minutes)')
    ax.set_ylabel('Label Value')
    ax.set_title('Label Timeline (Data Coverage)')
    ax.legend(loc='upper right')
    ax.set_yticks([0, 1, 2, 3, 4])
    
    plt.suptitle(f'Data Quality Visualization - Subject {data["subject"]}', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


# Visualize quality issues
visualize_quality_issues(subject_data)

### Data Quality Summary

#### Issues Identified

| Issue | Affected Signals | Severity | Mitigation Strategy |
|-------|------------------|----------|--------------------|
| **Motion Artifacts** | ECG, EMG | Medium | Butterworth bandpass filter, accelerometer-based detection |
| **Baseline Wander** | ECG, EDA | Medium | High-pass filter (0.5 Hz for ECG), polynomial detrending |
| **Powerline Noise** | ECG, EMG | Low | Notch filter (50/60 Hz) |
| **SCR Overlap** | EDA | Low | Advanced decomposition (cvxEDA) |
| **Sensor Drift** | Temperature | Low | Calibration, relative change features |
| **Unlabeled Periods** | All | High | Exclude label 0, 5, 6, 7 from analysis |

#### Recommended Preprocessing Pipeline

1. **ECG**: Bandpass (0.5-40 Hz) → Notch (50 Hz) → R-peak detection → HRV extraction
2. **EDA**: Low-pass (5 Hz) → Decomposition (tonic/phasic) → SCR detection
3. **EMG**: Bandpass (20-450 Hz) → Notch (50 Hz) → Rectification → Envelope
4. **Temperature**: Low-pass (0.1 Hz) → Relative change calculation
5. **Respiration**: Bandpass (0.1-0.5 Hz) → Peak detection → Breathing rate

---

## 8. Summary and Next Steps

### Key Findings

1. **Data Structure**: WESAD provides rich multimodal data from chest (700 Hz) and wrist (4-64 Hz) sensors
2. **Class Distribution**: Imbalanced classes require careful handling (stratification, class weights)
3. **Signal Characteristics**: Clear physiological differences between stress and baseline conditions
4. **Quality Issues**: Motion artifacts, baseline drift, and unlabeled periods need preprocessing

### Next Steps

1. **Notebook 02**: Signal preprocessing and filtering
2. **Notebook 03**: Feature extraction (time, frequency, nonlinear domains)
3. **Notebook 04**: Machine learning model training and LOSO evaluation
4. **Notebook 05**: Deep learning approaches (CNN, LSTM)
5. **Notebook 06**: Explainability analysis (SHAP, LIME)

### References

1. Schmidt, P., et al. (2018). "Introducing WESAD, a Multimodal Dataset for Wearable Stress and Affect Detection." ICMI 2018.
2. Makowski, D., et al. (2021). "NeuroKit2: A Python toolbox for neurophysiological signal processing."
3. Gjoreski, M., et al. (2017). "Monitoring stress with a wrist device using context." Journal of Biomedical Informatics.

In [ ]:
# Save session info for reproducibility
print("="*60)
print("SESSION INFORMATION")
print("="*60)
print(f"Subject analyzed: {subject_data['subject']}")
print(f"Demo mode: {DEMO_MODE}")
print(f"\nPackage versions:")
print(f"  NumPy: {np.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  Matplotlib: {plt.matplotlib.__version__}")
print(f"  SciPy: {signal.scipy.__version__}")